In [1]:
import os

from dotenv import load_dotenv

load_dotenv()  # Returns a boolean if the .env file was found and loaded 
%load_ext autoreload
%autoreload 2

In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# Test if the API key is working
llm.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 14, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c6b5b9a933', 'id': 'chatcmpl-E9iFQ0K4ZA79126LHjNQ46UrScxQl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fd4ed-3f40-7823-88c8-e255e338fb90-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 7, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Test Each Agent Individually

In [4]:
# Test retrieval
from src.rag.retrieve import retrieve_relevant_documents
from src.data.state import AgriChainState

test_state = {
    "complaint_text": "the tomatoes arrived bruised and moldy",
}

result = retrieve_relevant_documents(test_state)  # type: ignore

# Inspect 
for doc in result["retrieved_documents"]: # type: ignore
    print(doc["score"], doc["doc_type"], doc["content"][:80])

C:\Users\mjhog\MyCode\fullstack-academy\agrichain_project\src\rag\retrieve.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retrieval took 0.05 seconds
1.1126602 supplier_info Supplier SUP-5806 - Asia Region: Specializes in tomatoes. Quality score: 88/100.
1.1330247 supplier_info Supplier SUP-5814 - Europe Region: Specializes in tomatoes. Quality score: 76/10
1.1471378 supplier_info Supplier SUP-12905 - Europe Region: Specializes in tomatoes. Quality score: 66/1
1.3364317 supplier_info Supplier SUP-6479 Profile: Certified organic producer in Europe. Capacity: 218 t
1.4174404 supplier_info Supplier SUP-6050 Profile: Certified organic producer in North America. Capacity


The retrieval call returns supplier_info documents for customer quality complaints, as explained in 03_rag_knowledge_base,

In [5]:
import json
from collections import Counter

with open("../data/raw/knowledge_base.json") as f:
    kb = json.load(f)

print(Counter(doc["doc_type"] for doc in kb))

Counter({'supplier_info': 200, 'sop': 50, 'resolution_guide': 24})


In [ ]:
from src.models.classify import predict_category

# Create a test complaint
test_complaint_text = "the invoice charged me twice for the same order"

category = predict_category(test_complaint_text)  # type: ignore
print(f"Predicted category: {category}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Predicted category: Pricing Error



**Design note:** In this design the analyzer calls the trained ANN classifier first for category, then separately asks the LLM to assess severity.

In [7]:
# Test analyzer
from src.agents.analyzer import analyze_severity

test_state = {
    "complaint_text": "the tomatoes arrived bruised and moldy",
}
test_state_two = {
    "complaint_text": "metal shavings, someone could get hurt",
}
result = analyze_severity(test_state)
result_two = analyze_severity(test_state_two)
print("Severity:", result["severity"])
print("Reasoning:", result["severity_reasoning"])
print("Severity:", result_two["severity"])
print("Reasoning:", result_two["severity_reasoning"])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Severity: high
Reasoning: The complaint involves perishable goods (tomatoes) that are bruised and moldy, indicating they may spoil quickly and are no longer suitable for sale or consumption. This not only poses a financial loss for the company but also raises potential safety concerns for consumers. Additionally, the quality of the product directly impacts customer satisfaction and the relationship with the customer, making this a high-severity issue.
Severity: high
Reasoning: The presence of metal shavings poses a significant safety risk, as it could lead to injury if the goods are used or consumed. This raises immediate concerns for customer safety and potential liability for the company. While the financial impact may not be directly quantifiable, the risk of harm to customers can severely damage the company's reputation and customer trust, making this a high-severity issue.


In [8]:
# Test investigator
from src.agents.investigator import investigate

test_state = {
    "complaint_text": "the tomatoes arrived bruised and moldy",
}
result = investigate(test_state)
print("Retrieved documents:")
for doc in result["retrieved_documents"]:  # type: ignore
    print(doc["score"], doc["doc_type"], doc["content"][:80])

Retrieval took 0.00 seconds
Retrieved documents:
1.1126602 supplier_info Supplier SUP-5806 - Asia Region: Specializes in tomatoes. Quality score: 88/100.
1.1330247 supplier_info Supplier SUP-5814 - Europe Region: Specializes in tomatoes. Quality score: 76/10
1.1471378 supplier_info Supplier SUP-12905 - Europe Region: Specializes in tomatoes. Quality score: 66/1
1.3364317 supplier_info Supplier SUP-6479 Profile: Certified organic producer in Europe. Capacity: 218 t
1.4174404 supplier_info Supplier SUP-6050 Profile: Certified organic producer in North America. Capacity


In [9]:
# Test planner
from src.agents.planner import plan_resolution
from src.agents.analyzer import analyze_severity
from src.agents.investigator import investigate

test_state = {
    "complaint_text": "the tomatoes arrived bruised and moldy",
}
test_state = analyze_severity(test_state)
test_state = investigate(test_state)

result = plan_resolution(test_state)
print("Resolution plan:")
print(result["resolution_plan"])  # type: ignore

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Retrieval took 0.01 seconds
Resolution plan:
{'plan_summary': "To address the complaint regarding bruised and moldy tomatoes, we will initiate a refund for the affected order, investigate the supplier's quality control processes, and implement stricter quality checks for future shipments. Additionally, we will communicate with the customer to ensure their satisfaction and prevent similar issues in the future.", 'steps': ['Issue a full refund or replacement for the affected order of tomatoes.', 'Contact Supplier SUP-5806 to investigate the quality control measures in place and the conditions under which the tomatoes were shipped.', 'Review the shipping and handling processes to identify any potential issues that may have contributed to the bruising and mold.', 'Implement stricter quality checks before shipment, including visual inspections and temperature control measures during transport.', 'Communicate with the customer to inform them of the actio

In [10]:
# Test drafting a customer response
from src.agents.communicator import draft_customer_response
from src.agents.planner import plan_resolution
from src.agents.analyzer import analyze_severity
from src.agents.investigator import investigate

test_state = {
    "complaint_text": "the tomatoes arrived bruised and moldy",}
test_state = analyze_severity(test_state)
test_state = investigate(test_state)
test_state = plan_resolution(test_state)
result = draft_customer_response(test_state)

print("Customer response:")
print(result["customer_response"])  # type: ignore
print("error_messgae:", result.get("error_message", "No error message"))  # type: ignore

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Retrieval took 0.01 seconds
Customer response:
Subject: Our Apologies and Resolution for Your Recent Order

Dear [Customer's Name],

Thank you for reaching out to us regarding your recent order of tomatoes. We sincerely apologize for the inconvenience caused by the bruised and moldy condition of the tomatoes you received. We understand how disappointing this must be, and we appreciate you bringing this matter to our attention.

To address your concerns, we are taking the following steps:
1. **Full Refund**: We will initiate a full refund for your order of tomatoes immediately. You can expect to see this reflected in your account shortly.
2. **Supplier Investigation**: We are contacting our supplier to investigate their quality control measures and the delivery conditions that may have contributed to this issue. Ensuring the quality of our products is our top priority.
3. **Quality Assurance Protocol**: We are implementing stricter quality checks fo


**Design note:** Escalation only triggers on `severity == "critical"`, not `"high"` so escalation stays reserved for genuinely urgent cases.

In [11]:
# Test escalation manager
from src.agents.communicator import draft_customer_response
from src.agents.planner import plan_resolution
from src.agents.analyzer import analyze_severity
from src.agents.investigator import investigate
from src.agents.escalation_manager import identify_escalation

test_state = {
    "complaint_text": "the rotten tomatoes arrived in my order and I am now dead",
}
test_state = analyze_severity(test_state)
test_state = investigate(test_state)
test_state = plan_resolution(test_state)
test_state = draft_customer_response(test_state)
result = identify_escalation(test_state)
print("Escalation needed:", result["escalate"])  # type: ignore
print("Escalation reason:", result.get("escalation_reason", "No escalation reason"))  # type: ignore

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Retrieval took 0.01 seconds
Escalation needed: True
Escalation reason: Critical severity complaint


## Run the Full Compiled Graph
End-to-end testing

**Design note:** the graph is kept linear (no conditional edges) where `escalate` is stored as an internal boolean flag on the state rather than branching the graph itself.

In [12]:
from src.graph import app
import json

test_state = {
    "complaint_text": "the rotten tomatoes arrived in my order and I am now dead",
}

result = app.invoke(test_state)

print(json.dumps(result, indent=2, default=str))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Retrieval took 0.01 seconds
{
  "complaint_text": "the rotten tomatoes arrived in my order and I am now dead",
  "predicted_category": "Quality Issues",
  "severity": "critical",
  "severity_reasoning": "The complaint indicates that the customer received rotten tomatoes, which poses a significant health risk. The phrase 'I am now dead' suggests a serious safety concern, potentially implying severe allergic reactions or food poisoning. This situation requires immediate attention to prevent further harm and to address the customer's health and safety. Additionally, the financial impact could be substantial if the customer is seeking compensation or if this incident damages the company's reputation.",
  "retrieved_documents": [
    {
      "content": "Supplier SUP-5806 - Asia Region: Specializes in tomatoes. Quality score: 88/100. Average delivery time: 2 days. Known issues: excellent track record. Preferred for large volume orders.",
      "score": "

## Batch Testing on Real Complaints
Testing on 5 random test complaints

In [ ]:
import pandas as pd
import json
from src.graph import app

test_df = pd.read_csv("../data/raw/complaints_test.csv")

sample_complaints = test_df.sample(n=5, random_state=42)

reports = []
for index, row in sample_complaints.iterrows():
    result = app.invoke({"complaint_text": row["complaint_text"]})
    complaint_id = row["complaint_id"]
    actual_category = row["category"]
    actual_severity = row['priority']

    report_dict = {
        "complaint_id": complaint_id,
        "complaint_text": result.get("complaint_text"),
        "severity": result.get("severity"),
        "actual_severity": actual_severity,
        "predicted_category": result.get("predicted_category"),
        "actual_category": actual_category,
        "resolution_plan": result.get("resolution_plan"),
        "customer_response": result.get("customer_response"),
        "escalation_needed": result.get("escalate"),
        "escalation_reason": result.get("escalation_reason", "No escalation reason"),
    }
    reports.append(report_dict)

print("Reports:")
for report in reports:
    print(json.dumps(report, indent=2, default=str))

C:\Users\mjhog\MyCode\fullstack-academy\agrichain_project\src\models\embeddings.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
Retrieval took 0.01 seconds
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
Retrieval took 0.01 seconds
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
Retrieval took 0.01 seconds
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Retrieval took 0.01 seconds
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
Retrieval took 0.01 seconds
Reports:
{
  "complaint_id": "CMP-000014",
  "complaint_text": "Order ORD-891930 contains cucumbers that are already showing signs of spoilage upon arrival.",
  "severity": "high",
  "actual_severity": "critical",
  "predicted_category": "Quality Issues",
  "actual_category": "Quality Issues",
  "resolution_plan": {
    "plan_summary": "To address the complaint regarding the spoilage of cucumbers in order ORD-891930, we will initiate a thorough investigation into the supply chain process, communicate with the supplier, and offer a replacement or refund to the customer. We will also consider switching to a higher-rated supplier for future orders to ensure better

Saving the batch results to `data/processed/sample_resolution_reports.json` 


In [2]:
import json

with open("../data/processed/sample_resolution_reports.json", "w") as f:
    json.dump(reports, f, indent=2, default=str)

print(f"Saved {len(reports)} reports.")

Saved 5 reports.
